**Import Required Libraries**

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

**Load Project Utilities & Initialize Notebook Widgets**

In [0]:
%run /Workspace/consolidated_pipeline/1_setup/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sportsbar-dp-444115535128-sa-east-1-an/{data_source}/*.csv'
print(base_path)

## Bronze

In [0]:
df = (
    spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)

display(df.limit(100))

In [0]:
df.write\
    .format("delta")\
    .mode("overwrite")\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

## Silver

In [0]:
df_bronze = spark.read.table(f"{catalog}.{bronze_schema}.{data_source}")

display(df_bronze.limit(100))

df_bronze.printSchema()

**Transformations**

In [0]:
df_duplicates = df_bronze.groupBy("customer_id").count().filter(F.col("count") > 1)
display(df_duplicates)
df_duplicates.count()

In [0]:
print("Rows before duplicates dropped: ", df_bronze.count())
df_silver = df_bronze.dropDuplicates(["customer_id"])
print("Rows after duplicates dropped: ", df_silver.count())
display(df_silver.limit(100))

In [0]:
display(df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name"))))

In [0]:
df_silver = df_silver.withColumn("customer_name", F.trim(F.col("customer_name")))

In [0]:
display(df_silver.select("city").distinct())

In [0]:
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',
    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',
    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}

Allowed = {'Bengaluru', 'Hyderabad', 'New Delhi'}

df_silver = (df_silver.replace(city_mapping, subset=["city"])
                    .withColumn("city", F.when(F.col("city").isNull(), None).when(F.col("city").isin(Allowed), F.col("city")).otherwise(None)))

#Check
display(df_silver.select("city").distinct())

In [0]:
display(df_silver.select("customer_name").distinct())

In [0]:
df_silver = df_silver.withColumn("customer_name", F.when(F.col("customer_name").isNull(), None).otherwise(F.initcap("customer_name")))

In [0]:
display(df_silver.select("customer_name").distinct().orderBy("customer_name"))

In [0]:
city_nulls = df_silver.filter(F.col("city").isNull())

display(city_nulls)

In [0]:
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
display(df_silver.filter(F.col("customer_name").isin(null_customer_names)))

In [0]:
# Business confirmation note: City corrections confirmed by business team
customer_city_fix = {
    #Sprintx Nutrition
    789403: 'New Delhi',

    #Zenathlete Foods
    789420: 'Bengaluru',

    #Primefuel Nutrition
    789521: 'Hyderabad',

    #Recovery Lane
    789603: 'Hyderabad'
}

df_fix = spark.createDataFrame([(k,v) for k,v in customer_city_fix.items()], ["customer_id", "fixed_city"])

display(df_fix)

In [0]:
df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn("city", F.coalesce("city", "fixed_city"))
    .drop("fixed_city")
)

display(df_silver.select("city").distinct())

In [0]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
print(df_silver.printSchema())

### Standardizing Customer Attributes to Match Parent Company Data Model

In [0]:
df_silver = (
    df_silver
    # Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        F.concat_ws(
            "-",
            F.coalesce(F.col("customer_name"), F.lit("Unknown")),
            F.coalesce(F.col("city"), F.lit("Unknown"))
        )
    )

    # Static stributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

display(df_silver.limit(100))

In [0]:
df_silver.write\
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

## Gold

In [0]:
df_silver = spark.read.table(f"{catalog}.{silver_schema}.{data_source}")

# Take required columns only
# "customer_id, customer_name, city, read_timestamp, file_name, file_size, customer, market, platform, channel"
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

display(df_gold.limit(100))

In [0]:
df_gold.write\
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")

df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"), "customer", "market", "platform", "channel"
)

### Merging Data source with parent

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"), 
    condition="target.customer_code = source.customer_code"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()